<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

In [0]:
#| echo: false
#| output: asis
show_doc(leaf_stomatal_medlyn)

---

[source](https://github.com/ecamo19/plant_hydraulics/blob/main/plant_hydraulics/leaf_stomatal_medlyn.py#L15){target="_blank" style="float:right; font-size:smaller"}

### leaf_stomatal_medlyn

```python

def leaf_stomatal_medlyn(
    physcon:PhysCon, # Physical constants.
    atmos:Atmos, # Atmospheric forcing variables.
    leaf:Leaf, # Leaf parameters including:
- g0 : float
    Minimum stomatal conductance (mol H2O/m2/s).
- g1_medlyn : float
    Medlyn slope parameter (kPa^0.5).
- minl_wp : float
    Minimum leaf water potential (MPa) — the cavitation threshold.
    flux:Flux, # Flux variables. Must have tleaf, qa, psi_leaf, lsc, etc. set.
)->Flux: # Updated flux object with converged gs, An, Tleaf, psi_leaf, etc.


```

*Compute stomatal conductance using the Medlyn et al. (2011) model,*
coupled with leaf energy balance, photosynthesis, and plant hydraulics.

The Medlyn equation (Medlyn et al. 2011, Eq. 11) is:

    gs = g0 + 1.6 * (1 + g1 / sqrt(D)) * An / cs

where:

   - gs  = stomatal conductance to water vapor (mol H2O/m2/s)
   - g0  = minimum stomatal conductance (mol H2O/m2/s)
   - g1  = slope parameter (kPa^0.5)
   - D   = vapor pressure deficit at the leaf surface (kPa)
   - An  = net photosynthesis (umol CO2/m2/s)
   - cs  = CO2 concentration at the leaf surface (umol/mol)

The factor 1.6 converts from CO2 conductance to H2O conductance
(ratio of molecular diffusivities: D_H2O / D_CO2 = 1.6).

The coupling problem:
    gs depends on An (more photosynthesis → more stomatal opening)
    An depends on gs (wider stomata → more CO2 → more photosynthesis)
    Tleaf depends on gs (wider stomata → more transpiration → cooler leaf)
    An depends on Tleaf (enzyme kinetics change with temperature)

Solution approach — fixed-point iteration:

    1. Start with an initial guess for gs
    2. Compute Tleaf from energy balance (leaf_temperature)
    3. Compute An, cs, VPD from photosynthesis (leaf_photosynthesis)
    4. Update gs from Medlyn equation
    5. Repeat until gs converges

After convergence, a hydraulic safety check is applied:
    If the resulting leaf water potential drops below the minimum
    safe value (leaf.minl_wp), gs is iteratively reduced until
    hydraulic safety is restored. This prevents xylem cavitation.

 The Medlyn equation (Medlyn et al. 2011, Eq. 11):

   gs = g0 + 1.6 * (1 + g1 / sqrt(D)) * An / cs

where:

- g0 is the baseline conductance (always present)
- 1.6 represents the H2O/CO2 diffusivity ratio
- (1 + g1/sqrt(D)) represents the VPD sensitivity factor:

    - i.e. When D is small (humid): g1/sqrt(D) is large → gs is high
    - i.e. When D is large (dry):   g1/sqrt(D) is small → gs is low
    - g1 controls the magnitude of this VPD response

- (An / cs) represents the photosynthetic demand factor:
       - High An and low cs → stomata open wide for CO2
       - Low An (shade)     → little need to open stomata

- Example with g0 = 0.01, g1 = 4.45, D = 1.5 kPa, An = 10, cs = 350:

   gs = 0.01 + 1.6 * (1 + 4.45/sqrt(1.5)) * 10/350
   gs = 0.01 + 1.6 * (1 + 3.633) * 0.02857
   gs = 0.01 + 1.6 * 4.633 * 0.02857
   gs = 0.01 + 0.212
   gs = 0.222 mol H2O/m2/s

#### Example leaf_stomatal_medlyn()